# Understanding Semantic Retrieval with Vector Stores and Similarity Search

Welcome to the core module on advanced retrieval techniques. In modern Generative AI applications, simply querying a large language model (LLM) is insufficient; we must provide contextually relevant information from external knowledge bases. This process is known as Retrieval-Augmented Generation (RAG). At its heart of RAG lies the **retriever**, which acts as the intelligent gateway to your proprietary data.

This notebook introduces the fundamental mechanism powering advanced retrieval: **Similarity Search**. Instead of relying on keyword matching (like traditional database searches), similarity search converts both your knowledge documents and your user queries into high-dimensional numerical vectors (embeddings). These embeddings capture the *meaning* or *semantics* of the text. By calculating the distance (e.g., cosine similarity) between the query vector and all document vectors, we can accurately identify which pieces of information are conceptually related to the user's question, even if they don't share the same keywords.

Mastering this concept is critical for building robust RAG pipelines and complex state machines using LangGraph. By understanding how to initialize a vector store (like ChromaDB) and utilize its dedicated retriever interface, you learn the foundational skill of grounding LLM responses in verifiable facts. This knowledge prepares you to build sophisticated agents that dynamically decide *when* and *how* to retrieve context before generating an answer, moving beyond simple single-step chains into advanced, multi-stage reasoning graphs.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Conceptualize Embeddings:** Understand how text is converted into numerical vectors (embeddings) to capture semantic meaning.
*   **Implement Vector Stores:** Initialize and populate an in-memory vector database (ChromaDB) using LangChain components.
*   **Perform Similarity Search:** Execute both direct similarity searches (`similarity_search`) and structured retrieval calls (`retriever.invoke()`).
*   **Differentiate Retrieval Methods:** Understand the difference between calling a retriever directly versus accessing the underlying vector store for raw search functionality.
*   **Apply RAG Fundamentals:** Use semantic retrieval to accurately pull contextually relevant documents from a diverse knowledge base, forming the basis of an advanced RAG system.


### Setup and Imports

This cell imports necessary libraries for setting up the environment, handling secrets (`dotenv`), generating embeddings (using OpenAI), and managing a vector store (Chroma). These components are foundational for performing similarity search in RAG systems.


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


### 🔑 Environment Setup (Loading API Keys)

This cell uses `load_dotenv()` to load environment variables (like API keys) from a local `.env` file. This is crucial for securely accessing external services, such as OpenAI, without hardcoding sensitive credentials directly into the notebook.


In [3]:
# Load OPENAI_API_KEY from .env
load_dotenv()

True

### Document Initialization

This cell initializes a list of `Document` objects. Each document simulates a chunk of text retrieved from a knowledge base, containing both the core content (`page_content`) and associated metadata (e.g., `topic`). This structured data is essential for testing retrieval systems, allowing us to simulate diverse sources with specific attributes.


In [4]:
# Create a list of documents with metadata
docs = [
    Document(page_content="Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.",
             metadata={"topic": "space"}),
    Document(page_content="The International Space Station orbits Earth at about 400 km altitude and travels at 28,000 km/h.",
             metadata={"topic": "space"}),
    Document(page_content="Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.",
             metadata={"topic": "space"}),
    Document(page_content="NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from the Sun.",
             metadata={"topic": "space"}),
    Document(page_content="Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.",
             metadata={"topic": "space"}),
    Document(page_content="DNA is a double-helix molecule that carries the genetic instructions for all living organisms.",
             metadata={"topic": "biology"}),
    Document(page_content="Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.",
             metadata={"topic": "biology"}),
    Document(page_content="The Roman Empire at its peak covered over 5 million square kilometers across three continents.",
             metadata={"topic": "history"}),
    Document(page_content="The printing press, invented by Gutenberg around 1440, revolutionised the spread of knowledge.",
             metadata={"topic": "history"}),
    Document(page_content="The Amazon River discharges more freshwater into the ocean than any other river on Earth.",
             metadata={"topic": "geography"}),
    Document(page_content="The Sahara Desert spans about 9.2 million square kilometers across northern Africa.",
             metadata={"topic": "geography"}),
    Document(page_content="Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.",
             metadata={"topic": "biology"}),
]

### Code Explanation

This cell iterates through the list of loaded documents (`docs`) to display a preview of each document. It prints both the assigned index number and the metadata topic, followed by the full page content, allowing the user to visually inspect the retrieved context before proceeding with advanced RAG steps.


In [5]:
for ind, doc in enumerate(docs,1):
    # Print the document number (index) and its associated topic from the metadata.
    print(f"Doc No.: {ind} | Topic: {doc.metadata["topic"]}")
    # Print the actual text content of the document chunk.
    print(f"Content: {doc.page_content}\n")


Doc No.: 1 | Topic: space
Content: Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.

Doc No.: 2 | Topic: space
Content: The International Space Station orbits Earth at about 400 km altitude and travels at 28,000 km/h.

Doc No.: 3 | Topic: space
Content: Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.

Doc No.: 4 | Topic: space
Content: NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from the Sun.

Doc No.: 5 | Topic: space
Content: Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.

Doc No.: 6 | Topic: biology
Content: DNA is a double-helix molecule that carries the genetic instructions for all living organisms.

Doc No.: 7 | Topic: biology
Content: Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.

Doc No.: 8 | Topic: history
Content: The Roman Empire at its peak covered over 5 mi

### 📚 Code Explanation

This cell initializes and populates a ChromaDB vector store. It uses `OpenAIEmbeddings` to convert the raw text documents (`docs`) into numerical vectors (embeddings) and then stores these embeddings, along with their original texts, in an in-memory Chroma collection named `similarity_search_demo`. This step is crucial because it transforms unstructured data into a format optimized for fast semantic similarity search.


In [6]:
# Embed documents and store in an in-memory ChromaDB collection
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="similarity_search_demo",
)


In [7]:
print(len(vectorstore.get()["ids"]))

12


### Code Explanation

This cell configures a retriever object that uses semantic similarity search. The `vectorstore.as_retriever()` method wraps the vector store, allowing it to function as an LLM retriever. By setting `search_type="similarity"` and specifying `k=2`, we ensure the system retrieves the top 2 documents most semantically similar to the query based on cosine similarity.


In [17]:
# Similarity search retriever returns the k most semantically similar documents
# It ranks by cosine similarity between the query embedding and document embeddings

retriever = vectorstore.as_retriever(
    search_type="similarity",  # Defines the retrieval method: 'similarity' uses standard cosine distance.
    search_kwargs={"k": 2},   # Specifies that we want to retrieve the top k=2 most relevant documents.
)



### Similarity Search (Direct Retrieval)

This cell demonstrates a direct similarity search using the `vectorstore`. It takes a natural language query and retrieves the top $k$ most semantically similar documents from the stored vector embeddings. This is the foundational step of RAG, providing context for subsequent processing.


In [ ]:
# this method is not used generally inside chains, or if you require customizations then 
# also you do not use this method

query = "How do cells generate their energy?"

# Perform the similarity search on the vector store using the query and requesting k=3 results.
results = vectorstore.similarity_search(query, k=3)

print(f"{query}\n")
for i, doc in enumerate(results, 1):
    # Print result metadata (e.g., topic) and content for visualization.
    print(f"Result {i} [topic={doc.metadata['topic']}]")
    print(f"{doc.page_content}")
    print()



How do cells generate their energy?

Result 1 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

Result 2 [topic=biology]
Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.

Result 3 [topic=space]
Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.



### Code Explanation

This cell executes the core retrieval step of the RAG pipeline. It uses the `retriever` (which is a LangChain Runnable) to take the user's `query` and fetch relevant documents from the vector store. The loop then iterates through these retrieved documents, printing both their metadata (confirming the topic) and their content.


In [18]:
query = "How do rockets work?"

# retrievers in langchain are runnables, create chains using retrievers
results = retriever.invoke(query)

# All returned documents should be from the space topic
print(f"{query}\n")
for i, doc in enumerate(results, 1):
    # Print the result number and the document's metadata (e.g., topic)
    print(f"Result {i} [topic={doc.metadata['topic']}]")
    # Print the actual content of the retrieved document
    print(f"{doc.page_content}")
    print()


How do rockets work?

Result 1 [topic=space]
Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.

Result 2 [topic=space]
Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.

